In [42]:
import os
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey, Ed25519PublicKey
from dataclasses import dataclass
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey, X25519PublicKey
from cryptography.hazmat.primitives.ciphers.algorithms import AES256
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

@dataclass(frozen=True)
class CertificateSigningRequest:
    csr_id: bytes
    server_dsa_public_key: Ed25519PublicKey
    signature: bytes
    
@dataclass(frozen=True)
class Certificate:
    cert_id: bytes
    server_dsa_public_key: Ed25519PublicKey
    signature: bytes
    
@dataclass(frozen=True)
class Symetric:
    ciphertext: bytes
    iv: bytes
    tag: bytes
    ad: bytes

@dataclass(frozen=True)
class ServerMsg:
    cert: Certificate
    server_tls_public_key: X25519PublicKey
    hkdf_salt: bytes
    symetric: Symetric
    signatur: bytes
    

# @dataclass(frozen=True)
# class KeyExchangeClientMsg:
#     client_public_key: X25519PublicKey



In [43]:
# CA

ca_private_key = Ed25519PrivateKey.generate()
ca_public_key = ca_private_key.public_key()

In [44]:
# Server

server_dsa_private_key = Ed25519PrivateKey.generate()
server_dsa_public_key = server_dsa_private_key.public_key()
id = b"THIS_IS_THE_SERVER_CERT"
# Warum wir auch den public key signier oder allgmein alles was im zertifikat ist:
# 1. Duplicate Signature Key Selection Angriff: Der angreiffer kann so den public key austauschen, jetzt muss er aber noch ein public key finden
#    Damit die signautr stimmt, es gibt ein angriff daführ.
# 2. Mit der Signatur unterschreibe ich den "Vertrag", wen ich nur die id signier unterschreibe ich nur die id, jeder kann die weiter felder
#    manipulier wie er es will.
# 3. Was ist wen ich in eine ander Protokoll genau diesen Text "id" signier muss dan kann der angreifer die signatur kopieren und sie für das ander
#    Protokoll verwenden um seich einzulogen, wen ich aber alles signier gehört die signatur direkt den CSR
payload_to_sign = id + server_dsa_public_key.public_bytes_raw()

# The X25519 Key are not in the CSR: why we have to rotate it
server_csr = CertificateSigningRequest (
    csr_id=id,
    server_dsa_public_key=server_dsa_public_key,
    signature=server_dsa_private_key.sign(payload_to_sign)
)

In [45]:
# CA - signing cert

# Check server_cert - rais a error if fails
server_csr.server_dsa_public_key.verify(server_csr.signature, server_csr.csr_id + server_csr.server_dsa_public_key.public_bytes_raw())

cert = Certificate(
    cert_id=server_csr.csr_id,
    server_dsa_public_key=server_csr.server_dsa_public_key,
    signature=ca_private_key.sign(server_csr.csr_id + server_csr.server_dsa_public_key.public_bytes_raw())
)

In [46]:
# Client prepares message to server

client_private_key = X25519PrivateKey.generate()
client_public_key = client_private_key.public_key()

In [47]:
# Server accepts connection form client and send message

server_tls_private_key = X25519PrivateKey.generate()
server_tls_public_key = server_tls_private_key.public_key()
m = b"THIS IST THE RESPOSNE FORM THE SERVER"

shared_key = server_tls_private_key.exchange(client_public_key)
hkdf_salt = os.urandom(16)
hkdf = HKDF(algorithm=hashes.SHA3_256(), length=32, salt=hkdf_salt, info=b"shared_key")
key = hkdf.derive(shared_key)

iv = os.urandom(12)
ad = b'THIS ARE AUTENTICATED DATA BUT NOT ENCRYPTED'
encryptor = Cipher(algorithms.AES256(key), modes.GCM(iv)).encryptor()

encryptor.authenticate_additional_data(ad)
ciphertext = encryptor.update(m) + encryptor.finalize()
tag = encryptor.tag

payload = cert.signature + server_tls_public_key.public_bytes_raw() + hkdf_salt + ciphertext + iv + tag + ad
signatur = server_dsa_private_key.sign(payload)

symetric = Symetric(ciphertext, iv, tag, ad)

server_msg = ServerMsg(
    cert,
    server_tls_public_key,
    hkdf_salt,
    symetric,
    signatur
)

In [48]:
# Client encrypt message and validate cert

payload = server_msg.cert.cert_id + server_msg.cert.server_dsa_public_key.public_bytes_raw()
ca_public_key.verify(server_msg.cert.signature, payload)

payload = server_msg.cert.signature + \
    server_msg.server_tls_public_key.public_bytes_raw() + \
    server_msg.hkdf_salt + \
    server_msg.symetric.ciphertext + \
    server_msg.symetric.iv + \
    server_msg.symetric.tag +\
    server_msg.symetric.ad
cert.server_dsa_public_key.verify(server_msg.signatur, payload)

shared_key = client_private_key.exchange(server_msg.server_tls_public_key)
hkdf = HKDF(algorithm=hashes.SHA3_256(), length=32, salt=server_msg.hkdf_salt, info=b"shared_key")
key = hkdf.derive(shared_key)

decrytpor = Cipher(algorithms.AES256(key), mode=modes.GCM(server_msg.symetric.iv, server_msg.symetric.tag)).decryptor()

decrytpor.authenticate_additional_data(server_msg.symetric.ad)
dec_msg = decrytpor.update(server_msg.symetric.ciphertext) + decrytpor.finalize()


print(dec_msg)

b'THIS IST THE RESPOSNE FORM THE SERVER'
